In [2]:
import sys
sys.path.append('../src')

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from torchvision.models import vgg19
from skimage.color import lab2rgb
from tqdm import tqdm

#noinspection PyUnresolvedReferences
from model_resnet_unet_lab import ResNetUNet
#noinspection PyUnresolvedReferences
from dataset_lab import get_dataloader

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Устройство: {device}")

Устройство: cpu


In [4]:
BATCH_SIZE = 16
EPOCHS = 30
LEARNING_RATE = 1e-4  # Меньше стандартного (дообучение)
DATA_DIR = '../data/processed/fine-tuning'

# Веса модели для дообучения (если есть)
PRETRAINED_PATH = '../outputs/models/resnet_unet_lab_big.pth'
SAVE_DIR = '../outputs/models'

In [5]:
train_loader = get_dataloader(DATA_DIR, batch_size=BATCH_SIZE, shuffle=True)

print(f"Загружено {len(train_loader.dataset)} изображений")
print(f"Батчей: {len(train_loader)}")

Загружено 4813 изображений
Батчей: 301


In [ ]:
def lab_to_rgb(l_tensor, ab_tensor):
    """Преобразование тензоров Lab в RGB для отображения"""
    l = (l_tensor.detach().cpu().numpy() + 1.0) * 50.0
    ab = ab_tensor.detach().cpu().numpy() * 128.0
    lab = np.concatenate([l, ab], axis=0).transpose(1, 2, 0)
    rgb = lab2rgb(lab.astype(np.float64))
    return rgb

def show_prediction(model, l_batch, ab_batch, epoch):
    model.eval()
    with torch.no_grad():
        pred_ab = model(l_batch[:1].to(device))

    l_img = l_batch[0]
    real_ab = ab_batch[0]
    pred_ab = pred_ab[0]

    plt.figure(figsize=(12, 4))
    plt.subplot(1, 3, 1)
    plt.imshow(l_img[0].cpu(), cmap='gray')
    plt.title("Вход (L)")

    plt.subplot(1, 3, 2)
    plt.imshow(lab_to_rgb(l_img, pred_ab))
    plt.title(f"Результат (Эпоха {epoch})")

    plt.subplot(1, 3, 3)
    plt.imshow(lab_to_rgb(l_img, real_ab))
    plt.title("Оригинал")
    plt.show()

In [ ]:
class VisualLoss(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = vgg19(weights='IMAGENET1K_V1').features
        self.slice = nn.Sequential(*list(vgg.children())[:18]).eval()
        for param in self.slice.parameters():
            param.requires_grad = False

    def forward(self, fake_ab, real_ab, l_batch):
        fake_img = torch.cat([l_batch, fake_ab], dim=1)
        real_img = torch.cat([l_batch, real_ab], dim=1)
        return nn.functional.l1_loss(self.slice(fake_img), self.slice(real_img))

criterion_l1 = nn.L1Loss()
criterion_perceptual = VisualLoss().to(device)

In [ ]:
model = ResNetUNet(n_class=2).to(device)
model.load_state_dict(torch.load(PRETRAINED_PATH, map_location=device))
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, betas=(0.5, 0.999))

In [ ]:
history = {
    'total_loss': [],
    'l1_loss': [],
    'vgg_loss': []
}

print("\n🚀 НАЧИНАЕМ ДООБУЧЕНИЕ\n")

for epoch in range(EPOCHS):
    model.train()
    running_total = 0.0
    running_l1 = 0.0
    running_vgg = 0.0

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for l_batch, ab_batch in loop:
        l_batch, ab_batch = l_batch.to(device), ab_batch.to(device)

        optimizer.zero_grad()

        # Forward pass
        pred_ab = model(l_batch)

        # Вычисление потерь
        loss_l1 = criterion_l1(pred_ab, ab_batch)
        loss_vgg = criterion_perceptual(pred_ab, ab_batch, l_batch)
        total_loss = loss_l1 + 0.1 * loss_vgg

        # Backward pass
        total_loss.backward()
        optimizer.step()

        # Статистика
        running_total += total_loss.item()
        running_l1 += loss_l1.item()
        running_vgg += loss_vgg.item()

        loop.set_postfix(loss=total_loss.item())

    # Сохраняем средние значения за эпоху
    epoch_total = running_total / len(train_loader)
    epoch_l1 = running_l1 / len(train_loader)
    epoch_vgg = running_vgg / len(train_loader)

    history['total_loss'].append(epoch_total)
    history['l1_loss'].append(epoch_l1)
    history['vgg_loss'].append(epoch_vgg)

    print(f"✅ Эпоха {epoch+1} | Total: {epoch_total:.4f} | L1: {epoch_l1:.4f} | VGG: {epoch_vgg:.4f}")

    # Визуализация каждые 3 эпохи
    if (epoch + 1) % 3 == 0 or epoch == 0:
        l_batch, ab_batch = next(iter(train_loader))
        show_prediction(model, l_batch, ab_batch, epoch + 1, save=True)

    # Сохранение чекпоинтов каждые 5 эпох
    if (epoch + 1) % 5 == 0:
        torch.save(model.state_dict(), f'{SAVE_DIR}/resnet_unet_mixed_epoch_{epoch+1}.pth')
        print(f"   💾 Чекпоинт сохранён: epoch_{epoch+1}.pth")

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history['total_loss'], label='Total Loss', color='blue', linewidth=2)
plt.plot(history['l1_loss'], label='L1 Color Loss', linestyle='--', color='green')
plt.plot(history['vgg_loss'], label='VGG Perceptual Loss', linestyle='--', color='red')
plt.title('История потерь при дообучении')
plt.xlabel('Эпоха')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history['total_loss'][5:], color='blue', linewidth=2)  # Пропускаем первые 5
plt.title('Потери после начальной стабилизации')
plt.xlabel('Эпоха')
plt.ylabel('Total Loss')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/training_history.png', dpi=150)
plt.show()